<a href="https://colab.research.google.com/github/sohailpayami2023/digital-signal-processing-python/blob/main/notebooks/01_signal_fundamentals/01_sine_and_noise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Signal Fundamentals: Cosine, Noise, FFT, and Spectrogram

**Topics covered:**
- Generating and visualising real discrete-time sinusoidal signals
- Complex baseband (IQ) representation — the foundation of all wireless systems
- Modelling Additive White Gaussian Noise (AWGN) and controlling SNR
- Computing the FFT and interpreting the two-sided power spectrum
- Building and interpreting a spectrogram (STFT) and the time-frequency resolution trade-off

**Author:** Sohail Payami

## Libraries Used

| Library | What it does in this notebook |
|---------|-------------------------------|
| `numpy` (`np`) | Core numerical library — arrays, math functions, FFT. The Python equivalent of MATLAB built-in matrix operations. |
| `matplotlib.pyplot` (`plt`) | 2-D plotting — equivalent to MATLAB `plot`, `figure`, `subplot`. |
| `scipy.signal.chirp` | Generates a frequency-swept signal — equivalent to MATLAB `chirp`. |
| `scipy.signal.spectrogram` | Computes the Short-Time Fourier Transform (STFT) — equivalent to MATLAB `spectrogram`. |

The `plt.rcParams.update(...)` block sets global plot defaults for the whole notebook:
- `figure.dpi = 120` — higher resolution figures (MATLAB default is 72 dpi)
- `axes.grid = True` — grid on all plots by default
- `grid.alpha = 0.4` — semi-transparent grid lines
- `lines.linewidth = 1.2` — slightly thicker lines than default


In [ ]:
# In Python, libraries must be explicitly imported before use.
# Unlike MATLAB where toolboxes are always available, here you import only what you need.

import numpy as np                          # NumPy: Python's core numerical library.
                                            # 'as np' creates a short alias — so np.cos() instead of numpy.cos().
                                            # Equivalent to MATLAB's built-in matrix and math engine.

import matplotlib.pyplot as plt            # Matplotlib's pyplot module: Python's plotting library.
                                            # Equivalent to MATLAB's plot(), figure(), subplot(), etc.

from scipy.signal import chirp, spectrogram # Import specific functions from SciPy's signal module.
                                            # SciPy is Python's equivalent of MATLAB's Signal Processing Toolbox.
                                            # 'from X import Y' means: only bring Y into scope, not all of X.

# plt.rcParams is a global dictionary (key-value store) that controls Matplotlib's default settings.
# A dictionary in Python uses curly braces {} with 'key': value pairs — similar to a struct in MATLAB.
# rcParams.update({...}) applies multiple settings at once.
# MATLAB equivalent: set(groot,'DefaultAxesXGrid','on','DefaultLineLineWidth',1.2, ...)
plt.rcParams.update({
    'figure.dpi'     : 120,   # dots-per-inch — controls figure sharpness on screen (MATLAB default: 72)
    'axes.grid'      : True,  # show grid on every plot automatically (MATLAB: grid on)
    'grid.alpha'     : 0.4,   # grid line opacity: 0.0 = fully transparent, 1.0 = fully opaque
    'lines.linewidth': 1.2,   # default line thickness for all plots (MATLAB default: 0.5)
})


## 1. Real Discrete-Time Cosine Signal

The simplest test signal in DSP. In MATLAB you would write `t = 0:1/Fs:T-1/Fs;` and `x = cos(2*pi*f0*t);`.
NumPy's `arange` and `cos` are direct equivalents.

$$x[n] = A \cos\!\left(2\pi f_0 \frac{n}{F_s} + \phi\right), \quad n = 0, 1, \ldots, N-1$$

where $F_s$ is the sampling frequency (Hz) and $f_0$ is the tone frequency.
The **Nyquist criterion** requires $f_0 < F_s / 2$ — any frequency above half the sampling rate aliases back into the baseband.

In [ ]:
Fs = 10_000          # sampling frequency in Hz
                     # Python tip: underscores inside numbers are ignored by Python — 10_000 == 10000.
                     # They are just a readability aid, like a thousands separator.

f0 = 150             # tone frequency in Hz — well below the Nyquist limit of Fs/2 = 5000 Hz
duration = 0.1       # signal duration in seconds

# np.arange(start, stop, step) generates evenly spaced values.
# MATLAB equivalent: t = 0 : 1/Fs : duration-1/Fs
# Important: like MATLAB's colon operator, np.arange does NOT include the stop value.
t = np.arange(0, duration, 1/Fs)

# np.cos() and np.pi are direct equivalents of MATLAB's cos() and pi.
# NumPy operations apply element-wise to arrays automatically — no .* or ./ needed as in MATLAB.
x = np.cos(2 * np.pi * f0 * t)

N = len(t)           # len() returns the number of elements — MATLAB equivalent: length(t) or numel(t)

# f-strings (f'...') are Python's way to embed variables directly inside a string.
# MATLAB equivalent: fprintf('Samples: %d | Fs: %d Hz\n', N, Fs)
# Inside {}, you can write any Python expression. The // operator is integer (floor) division.
print(f'Samples: {N}  |  Fs: {Fs} Hz  |  Nyquist limit: {Fs//2} Hz  |  f0: {f0} Hz')


In [ ]:
# plt.subplots() creates a figure and returns TWO objects: the figure (fig) and the axes (ax).
# MATLAB equivalent: figure; ax = gca;
# figsize=(width, height) is a tuple — Python's immutable fixed-length sequence, written with ().
# Units are inches. MATLAB: set(gcf, 'Units', 'inches', 'Position', [x y width height])
fig, ax = plt.subplots(figsize=(8, 2.5))

# t * 1e3 converts seconds to milliseconds — same arithmetic as MATLAB, no .* needed for scalars.
ax.plot(t * 1e3, x)

# ax.set() sets multiple axes properties in one call using keyword arguments.
# This avoids writing ax.set_xlabel(), ax.set_ylabel(), ax.set_title() on separate lines.
# MATLAB equivalents: xlabel('...'), ylabel('...'), title('...')
ax.set(xlabel='Time (ms)', ylabel='Amplitude',
       title=f'Real Cosine — f0 = {f0} Hz, Fs = {Fs} Hz')

plt.tight_layout()  # automatically adjusts subplot spacing to prevent label overlap
                    # MATLAB equivalent: 'tight' in exportgraphics, or manual position adjustment
plt.show()          # renders and displays the figure — needed in some Python environments


## 2. Complex Baseband — The IQ Representation

In all modern wireless systems (LTE, 5G NR, Wi-Fi), signals are processed at **complex baseband** rather than as real bandpass waveforms.
The receiver down-converts the RF signal to produce two streams:

- **I (In-phase):** the real part
- **Q (Quadrature):** the imaginary part, 90° phase-shifted relative to I

A real bandpass cosine at carrier frequency $f_c$ has a complex baseband equivalent — a rotating **phasor**:

$$\tilde{x}[n] = e^{j 2\pi f_c n / F_s} = \underbrace{\cos(2\pi f_c n/F_s)}_{I} + j\underbrace{\sin(2\pi f_c n/F_s)}_{Q}$$

The imaginary unit in Python/NumPy is `1j` — **same syntax as MATLAB**.
In the IQ plane, this signal traces a perfect unit circle, rotating at $f_c$ revolutions per second.
The **envelope** $|\tilde{x}[n]| = 1$ is constant — there is no amplitude modulation here.

In [ ]:
# 1j is Python's imaginary unit — identical syntax to MATLAB's 1j (or 1i).
# ** is Python's power/exponent operator — MATLAB uses ^ for scalars, .^ for element-wise on arrays.
x_iq = np.exp(1j * 2 * np.pi * f0 * t)   # MATLAB: exp(1j*2*pi*f0*t)

# plt.subplots(3, 1, ...) creates a 3-row, 1-column grid of subplots.
# It returns 'axes' as a NumPy array: axes[0], axes[1], axes[2].
# sharex=True links all x-axes so zooming one panel zooms all. MATLAB: linkaxes(ax_array, 'x')
fig, axes = plt.subplots(3, 1, figsize=(8, 5), sharex=True)

# .real and .imag are Python attributes (no parentheses) that extract the real/imaginary part.
# MATLAB equivalents: real(x_iq) and imag(x_iq)
axes[0].plot(t * 1e3, x_iq.real, color='steelblue',  label='I  (real part)')
axes[1].plot(t * 1e3, x_iq.imag, color='darkorange', label='Q  (imaginary part)')
axes[2].plot(t * 1e3, np.abs(x_iq), color='green',   label='|I + jQ|  (envelope = 1)')

# A for loop over a list or array — MATLAB: for i = 1:length(axes) ... end
# Here 'ax' takes the value of each element in 'axes' one by one.
for ax in axes:
    ax.legend(loc='upper right')  # add legend — MATLAB: legend(...)
    ax.set_ylabel('Amplitude')
axes[2].set_xlabel('Time (ms)')
axes[0].set_title('Complex Baseband Signal — I, Q, and Envelope')
plt.tight_layout()
plt.show()

# subplot_kw passes a dictionary of settings to each Axes object at creation time.
# 'aspect': 'equal' ensures equal scaling on both axes so the circle is not distorted.
# MATLAB equivalent: axis equal
fig, ax = plt.subplots(figsize=(3.5, 3.5), subplot_kw={'aspect': 'equal'})

theta = np.linspace(0, 2*np.pi, 300)  # linspace: same as MATLAB linspace()

# 'k--' is a shorthand format string: 'k' = black, '--' = dashed line.
# Same convention as MATLAB: plot(x, y, 'k--')
# alpha controls transparency: 0 = invisible, 1 = fully opaque. No direct MATLAB equivalent.
ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, linewidth=0.8)

# [::5] is Python slice notation meaning 'every 5th element, from start to end'.
# MATLAB equivalent: x_iq.real(1:5:end)
ax.scatter(x_iq.real[::5], x_iq.imag[::5], s=8, color='steelblue', alpha=0.7)
ax.set(xlabel='I', ylabel='Q', title='Phasor in IQ Plane (every 5th sample)')
plt.tight_layout()
plt.show()


## 3. Additive White Gaussian Noise (AWGN)

AWGN is the standard model for thermal receiver noise. The received signal is:

$$r[n] = x[n] + w[n], \qquad w[n] \sim \mathcal{N}(0,\, \sigma^2)$$

Noise samples are **i.i.d.** (independent, identically distributed) Gaussian with zero mean and variance $\sigma^2$.
"White" means the power spectral density is flat across all frequencies — every frequency bin sees the same noise power.

**Signal-to-Noise Ratio:**

$$\text{SNR}_{\text{dB}} = 10\log_{10}\!\left(\frac{P_{\text{signal}}}{\sigma^2}\right)$$

To generate noise at a **target SNR**, we measure the signal power, derive the required $\sigma^2$, and scale `randn` output accordingly:

- MATLAB: `noise = sqrt(sigma2) * randn(1, N);`
- Python: `noise = np.sqrt(sigma2) * np.random.randn(N)`

At SNR = 0 dB the noise power equals the signal power. Below 0 dB the noise dominates.

In [ ]:
# 'def' defines a reusable function — MATLAB equivalent: function [out1, out2] = name(in1, in2)
# The triple-quoted string """...""" immediately after def is a docstring:
# Python's standard way to document functions. Access it with: help(add_awgn)
# MATLAB equivalent: the % comment lines above a function.
def add_awgn(signal, snr_db):
    """Add AWGN to a signal at a specified SNR (dB). Returns (noisy_signal, noise)."""
    signal_power = np.mean(np.abs(signal) ** 2)        # ** is element-wise power — MATLAB: .^
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise = np.sqrt(noise_power) * np.random.randn(len(signal))  # randn: same as MATLAB randn()
    # Python functions can return multiple values separated by a comma.
    # The caller receives them as a tuple: a, b = function()
    # MATLAB equivalent: [noisy, noise] = add_awgn(signal, snr_db)
    return signal + noise, noise

np.random.seed(42)  # fix the random seed so results are reproducible — MATLAB: rng(42)

# zip() pairs up two sequences element by element for use in a for loop.
# Here each 'ax' is paired with one SNR value from the list.
# MATLAB equivalent: for i = 1:3; ax = axes(i); snr = snr_list(i); ... end
fig, axes = plt.subplots(3, 1, figsize=(8, 6), sharex=True)
for ax, snr in zip(axes, [20, 10, 0]):
    x_noisy, _ = add_awgn(x, snr)
    # The underscore _ is a Python convention for 'I am intentionally ignoring this return value'.
    # Here we only need x_noisy, not the noise array itself.
    ax.plot(t * 1e3, x_noisy, linewidth=0.7)
    ax.set_ylabel('Amplitude')
    ax.set_title(f'SNR = {snr} dB')

# Negative index [-1] means the last element — MATLAB: axes(end)
axes[-1].set_xlabel('Time (ms)')

# plt.suptitle adds a title to the whole figure, above all subplots — MATLAB: sgtitle()
plt.suptitle('Cosine + AWGN at Different SNR Levels', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Numerically verify that the generated noise hits the target SNR.

# String multiplication: '-' * 30 repeats the character 30 times to make a separator line.
# No direct MATLAB equivalent — you would use repmat('-', 1, 30).
header = '  Target SNR  |  Measured SNR'
print(header)
print('-' * len(header))

for snr_target in [20, 10, 0, -5]:
    _, noise = add_awgn(x, snr_target)
    measured = 10 * np.log10(np.mean(x**2) / np.mean(noise**2))
    # f-string format specifier {value:>9.2f}:
    #   > means right-align, 9 is the total field width, .2f means 2 decimal places.
    # MATLAB equivalent: fprintf('%9.2f\n', measured)
    print(f'  {snr_target:>8} dB  |  {measured:>9.2f} dB')


## 4. Frequency Domain — FFT and Power Spectrum

The Fast Fourier Transform (FFT) decomposes the signal into its sinusoidal frequency components.
NumPy's `fft` is identical in behaviour to MATLAB's `fft`.

**Key steps — same as MATLAB:**
1. `np.fft.fft(x)` — compute the DFT (output is complex, length $N$)
2. `np.fft.fftshift(...)` — shift the zero-frequency bin to the centre (≡ MATLAB `fftshift`)
3. `np.fft.fftfreq(N, 1/Fs)` — construct the frequency axis in Hz

A real cosine at $f_0$ appears as **two impulses at $\pm f_0$** in the two-sided spectrum — one for each complex exponential in Euler's formula.
When AWGN is added, a **flat noise floor** appears across all bins; its height is determined by $\sigma^2 / N$.

**Amplitude in dBFS** (dB relative to full scale): $\; 20\log_{10}|X[k]/N|$

Normalising by $N$ ensures that the peak amplitude of a unit-amplitude cosine reads 0 dBFS, matching the `fft(x)/N` MATLAB convention.

In [ ]:
def compute_spectrum(sig, Fs):
    """Two-sided amplitude spectrum in dBFS, normalised by N."""
    N = len(sig)

    # np.fft.fft(): identical to MATLAB's fft()
    # np.fft.fftshift(): reorders the output so 0 Hz is at the centre — identical to MATLAB's fftshift()
    # Dividing by N normalises amplitude so a unit cosine gives 0 dBFS — same as fft(x)/N in MATLAB
    X     = np.fft.fftshift(np.fft.fft(sig)) / N

    # np.fft.fftfreq(N, d=1/Fs) returns frequency bin centres in Hz.
    # MATLAB equivalent: f = (-N/2 : N/2-1) * Fs/N
    freqs = np.fft.fftshift(np.fft.fftfreq(N, 1/Fs))

    # np.abs() computes magnitude of complex numbers — MATLAB: abs()
    # Adding 1e-12 before log10 prevents log(0) = -inf for bins with zero energy
    mag   = 20 * np.log10(np.abs(X) + 1e-12)
    return freqs, mag

np.random.seed(0)
x_noisy_10dB, _ = add_awgn(x, 10)

freqs, mag_clean = compute_spectrum(x, Fs)
_, mag_noisy     = compute_spectrum(x_noisy_10dB, Fs)

fig, axes = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
axes[0].plot(freqs, mag_clean)
axes[0].set(title='Amplitude Spectrum — Clean Signal', ylabel='Magnitude (dBFS)')

axes[1].plot(freqs, mag_noisy, linewidth=0.8)
axes[1].set(title='Amplitude Spectrum — Noisy Signal (SNR = 10 dB)',
             ylabel='Magnitude (dBFS)', xlabel='Frequency (Hz)')

for ax in axes:
    ax.set_xlim([-500, 500])  # set_xlim: equivalent to MATLAB's xlim([-500 500])
    # axvline draws a vertical reference line at a given x value — MATLAB equivalent: xline()
    ax.axvline( f0, color='red', linestyle='--', alpha=0.6, linewidth=1, label=f'+{f0} Hz')
    ax.axvline(-f0, color='red', linestyle='--', alpha=0.6, linewidth=1, label=f'-{f0} Hz')
    ax.legend()
plt.tight_layout()
plt.show()


## 5. Spectrogram — Time-Frequency Analysis

### Why the FFT alone is not enough

The FFT gives a **single, time-averaged snapshot** of frequency content — it cannot reveal *when* a frequency appeared or how the spectrum evolves over time.
Many real-world signals have time-varying spectral structure:

- A radar chirp sweeping from low to high frequency
- A 5G NR slot where DMRS pilots occupy specific OFDM symbols
- A frequency-hopped (FHSS) signal

The FFT collapses all of this time structure into one averaged picture, making the sweep invisible.

### Short-Time Fourier Transform (STFT)

The STFT solves this by sliding a short analysis window $w[\tau]$ along the signal and computing the FFT at each position $m$:

$$X[m, k] = \sum_{\tau=0}^{L-1} x[mH + \tau]\, w[\tau]\, e^{-j2\pi k\tau / N_{\text{FFT}}}$$

where $L$ is the **window length**, $H$ is the **hop size** (stride between windows), and $k$ is the frequency bin index.
The result is a 2-D array indexed by time frame $m$ and frequency bin $k$.

The **spectrogram** is the squared magnitude:

$$S[m, k] = |X[m, k]|^2$$

### Time-frequency trade-off

There is a fundamental uncertainty principle (analogous to Heisenberg in quantum mechanics) that limits simultaneous time and frequency resolution:

$$\Delta t \cdot \Delta f \geq \frac{1}{4\pi}$$

| Window length | Time resolution | Frequency resolution |
|:---:|:---:|:---:|
| Short (e.g. 64 samples) | Fine — captures rapid changes | Poor — wide frequency bins |
| Long (e.g. 1024 samples) | Poor — temporal blurring | Fine — narrow frequency bins |

This is the same trade-off behind the 5G NR **subcarrier spacing (SCS)** choice:
15 kHz SCS uses a long OFDM symbol (~66.7 µs, fine frequency resolution), while 120 kHz SCS uses a short symbol (~8.3 µs, better time resolution for high-mobility channels).

### Chirp signal for demonstration

A static tone produces a flat horizontal line in the spectrogram — it teaches nothing about time-frequency structure.
Instead we use a **linear chirp** whose instantaneous frequency increases linearly with time:

$$f_i(t) = f_{\text{start}} + \frac{f_{\text{end}} - f_{\text{start}}}{T}\, t$$

This makes the time-varying structure immediately visible, and the trade-off between time and frequency resolution easy to observe.

In [ ]:
Fs_ch = 8_000
T_ch  = 1.0

# np.linspace(start, stop, num) — same as MATLAB linspace(start, stop, num)
# endpoint=False excludes the stop value, avoiding a duplicate sample at the period boundary.
# int() converts the float result of Fs_ch * T_ch to an integer — Python array sizes must be integers.
t_ch = np.linspace(0, T_ch, int(Fs_ch * T_ch), endpoint=False)

# Python allows assigning multiple variables in one line using tuple unpacking.
# MATLAB equivalent: f_start = 50; f_end = 2000;
f_start, f_end = 50, 2000

# scipy.signal.chirp uses keyword arguments (f0=..., f1=..., t1=..., method=...).
# Keyword arguments can be passed in any order — unlike MATLAB's positional-only arguments.
# MATLAB equivalent: chirp(t_ch, f_start, T_ch, f_end)
x_chirp = chirp(t_ch, f0=f_start, f1=f_end, t1=T_ch, method='linear')

np.random.seed(7)
x_chirp_noisy, _ = add_awgn(x_chirp, 20)

fig, ax = plt.subplots(figsize=(8, 2.5))
ax.plot(t_ch, x_chirp_noisy, linewidth=0.6)
ax.set(xlabel='Time (s)', ylabel='Amplitude',
       title=f'Linear Chirp: {f_start} Hz to {f_end} Hz over {T_ch} s  (SNR = 20 dB)')
plt.tight_layout()
plt.show()


In [ ]:
# Side-by-side comparison: FFT (time-averaged) vs spectrogram (time-resolved)
nperseg  = 256           # window length in samples: 256 / 8000 Hz = 32 ms per frame

# // is Python's integer (floor) division operator — MATLAB: floor(nperseg / 2)
noverlap = nperseg // 2

# scipy.signal.spectrogram() returns three arrays as a tuple: (frequencies, times, power_matrix)
# Tuple unpacking assigns all three in one line — MATLAB: [s, f, t] = spectrogram(...)
f_sg, t_sg, Sxx = spectrogram(x_chirp_noisy, fs=Fs_ch,
                               nperseg=nperseg, noverlap=noverlap, window='hann')

freqs_ch, mag_ch = compute_spectrum(x_chirp_noisy, Fs_ch)

fig, axes = plt.subplots(3, 1, figsize=(8, 8))

axes[0].plot(t_ch, x_chirp_noisy, linewidth=0.5)
axes[0].set(xlabel='Time (s)', ylabel='Amplitude', title='Time Domain — Chirp Signal')

# Boolean masking: (freqs_ch >= 0) creates a True/False array of the same length as freqs_ch.
# Using it as an index selects only the elements where the condition is True.
# MATLAB equivalent: freqs_ch(freqs_ch >= 0)
pos_mask = freqs_ch >= 0
axes[1].plot(freqs_ch[pos_mask], mag_ch[pos_mask])
axes[1].set(xlabel='Frequency (Hz)', ylabel='Magnitude (dBFS)',
             title='FFT — frequency content collapsed over all time (sweep structure is lost)',
             xlim=[0, 2500])

# pcolormesh displays a 2D matrix as a colour-mapped image — MATLAB equivalent: imagesc() or pcolor()
# shading='gouraud' smooths colour transitions between grid cells
# cmap='viridis' sets the colourmap — also available in MATLAB R2015b+: colormap(viridis)
im = axes[2].pcolormesh(t_sg, f_sg, 10 * np.log10(Sxx + 1e-12),
                         shading='gouraud', cmap='viridis')
axes[2].set(xlabel='Time (s)', ylabel='Frequency (Hz)',
             title=f'Spectrogram (STFT) — window = {nperseg} samples ({nperseg/Fs_ch*1e3:.0f} ms), 50% overlap',
             ylim=[0, 2500])

# colorbar adds a colour scale on the side of the plot — MATLAB: colorbar
# The 'label' keyword sets the colorbar axis label
fig.colorbar(im, ax=axes[2], label='Power (dB)')
plt.tight_layout()
plt.show()


In [ ]:
# Visualise the time-frequency trade-off by varying the analysis window length.

# plt.subplots(1, 3, ...) creates 1 row and 3 columns of subplots.
# sharey=True links all y-axes together — MATLAB: linkaxes(ax_array, 'y')
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)

# zip(axes, [...]) pairs each subplot axes object with one window size.
# This is a clean Python alternative to a MATLAB index-based loop:
# MATLAB: for i = 1:3; nper = win_sizes(i); ax = axes(i); ... end
for ax, nper in zip(axes, [64, 256, 1024]):
    f_s, t_s, S = spectrogram(x_chirp, fs=Fs_ch, nperseg=nper,
                               noverlap=nper // 2, window='hann')
    ax.pcolormesh(t_s, f_s, 10 * np.log10(S + 1e-12), shading='gouraud', cmap='viridis')
    freq_res    = Fs_ch / nper          # frequency resolution in Hz
    time_res_ms = nper / Fs_ch * 1e3   # time resolution in milliseconds
    ax.set(title=f'Window = {nper} samples ({time_res_ms:.1f} ms) | df={freq_res:.0f} Hz  dt={time_res_ms:.1f} ms',
           xlabel='Time (s)', ylim=[0, 2500])

axes[0].set_ylabel('Frequency (Hz)')
plt.suptitle('Time-Frequency Trade-off: Short window gives fine time resolution; Long window gives fine frequency resolution',
             fontsize=10, y=1.02)
plt.tight_layout()
plt.show()


## MATLAB to Python / NumPy Quick Reference

| Operation | MATLAB | Python (NumPy / SciPy) |
|-----------|--------|------------------------|
| Time vector | `t = 0:1/Fs:T-1/Fs;` | `t = np.arange(0, T, 1/Fs)` |
| Cosine | `cos(2*pi*f0*t)` | `np.cos(2*np.pi*f0*t)` |
| Complex exponential | `exp(1j*2*pi*f0*t)` | `np.exp(1j*2*np.pi*f0*t)` |
| Real / imaginary part | `real(x)` / `imag(x)` | `x.real` / `x.imag` |
| Envelope | `abs(x)` | `np.abs(x)` |
| White Gaussian noise | `randn(1, N)` | `np.random.randn(N)` |
| Signal power | `mean(abs(x).^2)` | `np.mean(np.abs(x)**2)` |
| FFT | `fft(x)` | `np.fft.fft(x)` |
| Centred FFT | `fftshift(fft(x))` | `np.fft.fftshift(np.fft.fft(x))` |
| Frequency axis (Hz) | `(-N/2:N/2-1)*Fs/N` | `np.fft.fftshift(np.fft.fftfreq(N, 1/Fs))` |
| dB magnitude | `20*log10(abs(X))` | `20*np.log10(np.abs(X))` |
| Linear chirp | `chirp(t, f0, T, f1)` | `scipy.signal.chirp(t, f0, f1, t1)` |
| Spectrogram | `spectrogram(x, win, nov, nfft, Fs)` | `scipy.signal.spectrogram(x, fs=Fs, nperseg=L, noverlap=H)` |